# 10 | LangGraph 的三种绘图方式

学习 LangGraph 时，画图不是为了好看，而是为了快速看清三件事：节点有哪些、边怎么连接、条件分支会走向哪里。

这篇文章只围绕一个问题：如何把同一个 LangGraph 画出来？

下面会用同一个示例，分别演示三种方式：

| 方式 | 适合场景 |
| --- | --- |
| Graphviz PNG | 本地渲染，适合实际业务配图和导出图片 |
| Mermaid PNG | 默认样式清爽，适合 Notebook 中快速预览 |
| ASCII 文本图 | 不需要图片环境，适合终端或调试 |

如果只是学习和写文章，我会优先用 `draw_mermaid_png()`；如果想本地稳定导出图片，再用 Graphviz。


## 一、准备绘图环境

LangGraph 自己负责描述图结构，真正渲染图片时通常会借助外部工具。

| 组件 | 作用 |
| --- | --- |
| LangGraph | 提供节点和边 |
| Graphviz | 本地绘图程序 |
| PyGraphviz | Python 调用 Graphviz 的接口 |
| grandalf | 绘制 ASCII 文本图 |

安装命令：

```bash
brew install graphviz
uv add pygraphviz grandalf
```

检查 Graphviz 是否安装成功：

```bash
dot -V
```

如果安装 `pygraphviz` 时提示找不到 `graphviz/cgraph.h`，可以显式指定 Graphviz 路径：

```bash
CFLAGS="-I$(brew --prefix graphviz)/include" \
LDFLAGS="-L$(brew --prefix graphviz)/lib" \
uv add pygraphviz
```


## 二、准备一个用于绘图的 LangGraph

为了看清分支和循环，下面用一个“内容生成 Agent”的流程作为示例：

1. 先理解用户问题；
2. 判断是否需要检索资料；
3. 可以直接回答，也可以先检索再生成草稿；
4. 最后进入审核；
5. 审核不通过时，回到修改节点，形成循环。


In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    query: str
    needs_research: bool
    approved: bool
    answer: str


# 下面这些节点函数不是为了真正执行 Agent，
# 而是为了构造一个有分支、有循环的 LangGraph。
def understand(state: State):
    return {}


def classify(state: State):
    return {}


def direct_answer(state: State):
    return {"answer": "直接回答"}


def retrieve(state: State):
    return {}


def draft(state: State):
    return {"answer": "根据资料生成草稿"}


def review(state: State):
    return {}


def revise(state: State):
    return {"answer": "修改后的答案"}


def publish(state: State):
    return {}


# 条件边：根据状态决定走哪条分支。
def route_request(state: State) -> Literal["direct_answer", "retrieve"]:
    return "retrieve" if state["needs_research"] else "direct_answer"


# 条件边：审核通过就发布，否则继续修改。
def route_review(state: State) -> Literal["publish", "revise"]:
    return "publish" if state["approved"] else "revise"


builder = StateGraph(State)

# 添加节点。
builder.add_node("understand", understand)
builder.add_node("classify", classify)
builder.add_node("direct_answer", direct_answer)
builder.add_node("retrieve", retrieve)
builder.add_node("draft", draft)
builder.add_node("review", review)
builder.add_node("revise", revise)
builder.add_node("publish", publish)

# 添加普通边。
builder.add_edge(START, "understand")
builder.add_edge("understand", "classify")
builder.add_edge("direct_answer", "review")
builder.add_edge("retrieve", "draft")
builder.add_edge("draft", "review")
builder.add_edge("revise", "review")
builder.add_edge("publish", END)

# 添加条件边。
builder.add_conditional_edges(
    "classify",
    route_request,
    {"direct_answer": "direct_answer", "retrieve": "retrieve"},
)
builder.add_conditional_edges(
    "review",
    route_review,
    {"publish": "publish", "revise": "revise"},
)

# 编译后得到可以绘图的 graph。
graph = builder.compile()


## 三、方式一：Graphviz PNG

`draw_png()` 会使用本地 Graphviz 渲染 PNG。它适合需要稳定导出图片的场景，例如写文章、做课件、保存到本地。


In [ ]:
from IPython.display import Image, display

# None 表示不保存文件，直接返回 PNG 二进制数据。
png = graph.get_graph().draw_png(None)
display(Image(png))


## 四、方式二：Mermaid PNG

`draw_mermaid_png()` 是 Notebook 里最省事的看图方式。它不需要自己写 Graphviz 样式，默认效果也比较清爽，适合学习时快速预览图结构。


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


## 五、方式三：ASCII 文本图

`draw_ascii()` 会把图画成文本，适合在终端或日志里快速粗看结构。

它的优点是简单：不需要图片显示环境，也方便复制到纯文本里。

它的缺点也很明显：复杂图会变形，条件分支标签看不到，循环边也不直观。所以它更适合调试，不适合作为文章里的正式配图。

如果报错提示缺少 `grandalf`，安装即可：

```bash
uv add grandalf
```


In [ ]:
print(graph.get_graph().draw_ascii())


## 六、怎么选择

简单记法：

| 需求 | 推荐方式 |
| --- | --- |
| Notebook 里快速看图 | Mermaid PNG |
| 业务中导出正式图片 | Graphviz PNG |
| 终端里快速看结构 | ASCII 文本图 |

对我来说，学习 LangGraph 时最常用的是 Mermaid PNG；等文章需要固定配图时，再切到 Graphviz PNG。
